# Redrob Hackathon - Role 2: ML Ranking Pipeline

This notebook runs the full **Indexer + Scorer** pipeline on the 100K candidate dataset.

**Instructions:**
1. Upload `candidates.jsonl` (487 MB) to this Colab session (or mount Google Drive).
2. Run all cells.
3. Download the 4 generated artifacts at the end.

> **GPU Tip:** Go to `Runtime > Change runtime type > T4 GPU` for ~5 min embedding time instead of 21 hours on CPU.

## 1. Install Dependencies

In [ ]:
!pip install -q numpy faiss-cpu sentence-transformers rank_bm25

## 2. Upload Dataset

Run this cell to upload `candidates.jsonl` from your local machine. Or skip and use Google Drive mount below.

In [ ]:
import os

# Option A: Upload from local machine
if not os.path.exists('candidates.jsonl'):
    from google.colab import files
    uploaded = files.upload()  # Upload candidates.jsonl here

# Option B: Mount Google Drive (uncomment below if file is in Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/candidates.jsonl' ./candidates.jsonl

print(f"File size: {os.path.getsize('candidates.jsonl') / 1024 / 1024:.1f} MB")

## 3. Config (All Constants)

In [ ]:
# =============================================================================
# JD QUERIES
# =============================================================================

SEMANTIC_JD_QUERY = """
Senior AI Engineer with 6 to 8 years of applied machine learning experience at product companies.
Built and shipped end-to-end ranking systems, search engines, and recommendation systems to real users at scale.
Production experience with embeddings-based retrieval using sentence-transformers, BGE, or E5 models.
Hands-on with vector databases and hybrid search infrastructure including FAISS, Pinecone, Weaviate, Qdrant, Milvus, or Elasticsearch.
Designed evaluation frameworks for ranking quality using NDCG, MRR, MAP, precision, and A/B testing.
Strong Python engineer who writes production-quality code, not just prototypes.
Experience with LLM integration, fine-tuning with LoRA and PEFT, and retrieval-augmented generation.
Worked at startups or product-focused technology companies, not IT consulting or services firms.
Understands the full lifecycle of building ML systems from offline training to online serving and monitoring.
"""

BM25_JD_QUERY = """
AI engineer machine learning ranking retrieval recommendation search
embeddings sentence-transformers FAISS Pinecone Weaviate Qdrant Milvus Elasticsearch
vector database hybrid search semantic search information retrieval
NDCG MRR MAP precision recall evaluation A/B testing
Python PyTorch TensorFlow scikit-learn production deployment
LLM fine-tuning LoRA QLoRA PEFT RAG retrieval-augmented generation
NLP natural language processing text classification named entity recognition
product company startup SaaS platform scale
data pipeline feature engineering model serving inference optimization
"""

# =============================================================================
# MODEL & SCORING
# =============================================================================

EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"
EMBEDDING_DIMENSION = 768
SEMANTIC_WEIGHT = 0.65
BM25_WEIGHT = 0.35

# =============================================================================
# CONSULTING FIRMS
# =============================================================================

CONSULTING_FIRMS = {
    'tcs', 'tata consultancy', 'infosys', 'wipro', 'accenture',
    'cognizant', 'capgemini', 'hcl', 'tech mahindra', 'mindtree',
    'mphasis', 'l&t infotech', 'lti', 'ltimindtree', 'hexaware',
    'cyient', 'persistent systems', 'zensar', 'niit technologies',
    'birlasoft', 'coforge', 'sonata software', 'mastek',
}

# =============================================================================
# TITLE TIERS
# =============================================================================

TITLE_TIER_PERFECT = {
    'ai engineer', 'ml engineer', 'machine learning engineer',
    'senior ai engineer', 'senior ml engineer', 'senior machine learning engineer',
    'lead ai engineer', 'lead ml engineer', 'staff ai engineer',
    'data scientist', 'senior data scientist', 'lead data scientist',
    'nlp engineer', 'senior nlp engineer',
    'recommendation systems engineer', 'search engineer',
    'ranking engineer', 'applied scientist', 'research engineer',
    'junior ai engineer', 'junior ml engineer',
}

TITLE_TIER_STRONG_ADJACENT = {
    'software engineer', 'senior software engineer', 'staff software engineer',
    'backend engineer', 'senior backend engineer',
    'full stack developer', 'senior full stack developer',
    'data engineer', 'senior data engineer',
    'platform engineer', 'infrastructure engineer',
    'devops engineer', 'senior devops engineer',
    '.net developer', 'java developer', 'python developer',
}

TITLE_TIER_WEAK_ADJACENT = {
    'project manager', 'senior project manager',
    'business analyst', 'senior business analyst',
    'product manager', 'senior product manager',
    'qa engineer', 'senior qa engineer', 'test engineer',
    'cloud engineer', 'senior cloud engineer',
    'frontend engineer', 'senior frontend engineer',
    'mobile developer', 'senior mobile developer',
    'technical lead', 'engineering manager',
}

TITLE_SCORES = {
    'perfect': 1.0,
    'strong_adjacent': 0.85,
    'weak_adjacent': 0.5,
    'non_tech_trap': 0.05,
}

# =============================================================================
# EXPERIENCE RANGE
# =============================================================================

EXPERIENCE_SCORES = [
    (5, 9, 1.0),
    (4, 5, 0.85),
    (9, 12, 0.85),
    (3, 4, 0.6),
    (12, 15, 0.6),
    (0, 3, 0.3),
    (15, 100, 0.3),
]

# =============================================================================
# LOCATION SCORING
# =============================================================================

PREFERRED_CITIES_TIER1 = {
    'pune', 'noida', 'delhi', 'new delhi', 'gurgaon', 'gurugram',
    'faridabad', 'ghaziabad', 'greater noida',
}

PREFERRED_CITIES_TIER2 = {
    'hyderabad', 'mumbai', 'bangalore', 'bengaluru', 'chennai',
}

LOCATION_SCORES = {
    'india_tier1': 1.3,
    'india_tier2': 1.15,
    'india_other_relocate': 1.0,
    'india_other_no_relocate': 0.7,
    'outside_india_relocate': 0.5,
    'outside_india_no_relocate': 0.2,
}

# =============================================================================
# HONEYPOT THRESHOLDS
# =============================================================================

HONEYPOT_DATE_MISMATCH_TOLERANCE = 0.5
HONEYPOT_CAREER_OVERLAP_DAYS = 60
HONEYPOT_EXPERT_COUNT_THRESHOLD = 8
HONEYPOT_EXPERT_AVG_DURATION_MIN = 6
HONEYPOT_EXPERIENCE_RATIO = 1.5

# =============================================================================
# ARTIFACT FILE PATHS
# =============================================================================

DENSE_INDEX_PATH = "dense_index.faiss"
BM25_INDEX_PATH = "bm25_index.pkl"
CANDIDATES_META_PATH = "candidates_meta.pkl"
JD_VECTOR_PATH = "jd_vector.npy"

print("Config loaded.")

## 4. Indexer (Offline Pre-computation)

This cell generates all 4 artifacts: FAISS index, BM25 index, lean metadata, and JD vector.

- **GPU (T4):** ~5 minutes
- **CPU:** ~45-60 minutes

In [ ]:
import json
import re
import os
import pickle
import time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi


def extract_text(candidate: dict) -> str:
    """Build text representation for embedding."""
    profile = candidate.get("profile", {})
    parts = [
        profile.get("headline", ""),
        profile.get("summary", ""),
    ]
    for skill in candidate.get("skills", []):
        name = skill.get("name", "")
        prof = skill.get("proficiency", "")
        if name:
            parts.append(f"{name} ({prof})" if prof else name)
    for role in candidate.get("career_history", [])[:3]:
        title = role.get("title", "")
        company = role.get("company", "")
        desc = role.get("description", "")
        if title:
            parts.append(f"{title} at {company}" if company else title)
        if desc:
            parts.append(desc)
    return " ".join(filter(None, parts))


def tokenize(text: str) -> list:
    return re.findall(r"\b\w+\b", text.lower())


def extract_lean_meta(candidate: dict) -> dict:
    """Extract only fields needed during 5-min inference."""
    profile = candidate.get("profile", {})
    return {
        "candidate_id": candidate["candidate_id"],
        "career_history": candidate.get("career_history", []),
        "skills": candidate.get("skills", []),
        "profile": {
            "current_title": profile.get("current_title", ""),
            "years_of_experience": profile.get("years_of_experience", 0),
            "location": profile.get("location", ""),
            "country": profile.get("country", ""),
            "headline": profile.get("headline", ""),
            "summary": profile.get("summary", "")[:200],
            "current_company": profile.get("current_company", ""),
            "current_industry": profile.get("current_industry", ""),
        },
        "redrob_signals": candidate.get("redrob_signals", {}),
    }


# ── Load candidates ──────────────────────────────────────────────────────────
INPUT_FILE = "candidates.jsonl"
OUTPUT_DIR = "."

candidates = []
texts = []

print(f"Loading {INPUT_FILE} ...")
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            c = json.loads(line)
            candidates.append(c)
            texts.append(c.get("text", extract_text(c)))

print(f"[OK] Loaded {len(candidates)} candidates")

# ── Generate embeddings ───────────────────────────────────────────────────────
print(f"Loading model: {EMBEDDING_MODEL_NAME} ...")
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Encoding candidate texts (this is the heavy step) ...")
start = time.time()
candidate_vectors = model.encode(
    texts, batch_size=256, show_progress_bar=True, normalize_embeddings=True
)
elapsed = time.time() - start
print(f"Encoding completed in {elapsed/60:.1f} minutes")

print("Encoding JD semantic query ...")
jd_vector = model.encode([SEMANTIC_JD_QUERY], normalize_embeddings=True)

candidate_vectors = np.ascontiguousarray(candidate_vectors, dtype=np.float32)
jd_vector = np.ascontiguousarray(jd_vector, dtype=np.float32)

# ── Build FAISS index ─────────────────────────────────────────────────────────
print("Building FAISS IndexFlatIP ...")
index = faiss.IndexFlatIP(EMBEDDING_DIMENSION)
index.add(candidate_vectors)
faiss.write_index(index, os.path.join(OUTPUT_DIR, DENSE_INDEX_PATH))
np.save(os.path.join(OUTPUT_DIR, JD_VECTOR_PATH), jd_vector)

# ── Build BM25 index ──────────────────────────────────────────────────────────
print("Building BM25 index ...")
tokenized_corpus = [tokenize(text) for text in texts]
bm25 = BM25Okapi(tokenized_corpus)
with open(os.path.join(OUTPUT_DIR, BM25_INDEX_PATH), "wb") as f:
    pickle.dump(bm25, f)

# ── Save lean metadata ────────────────────────────────────────────────────────
print("Saving lean candidate metadata ...")
lean_meta = [extract_lean_meta(c) for c in candidates]
with open(os.path.join(OUTPUT_DIR, CANDIDATES_META_PATH), "wb") as f:
    pickle.dump(lean_meta, f)

# ── Report sizes ──────────────────────────────────────────────────────────────
print("\n-- Artifact Sizes --")
total = 0
for name in [DENSE_INDEX_PATH, BM25_INDEX_PATH, CANDIDATES_META_PATH, JD_VECTOR_PATH]:
    path = os.path.join(OUTPUT_DIR, name)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        total += size_mb
        print(f"  {name:30s}  {size_mb:8.1f} MB")
print(f"  {'TOTAL':30s}  {total:8.1f} MB")
print(f"  Within 5 GB limit: {total / 5000 * 100:.1f}% used")

# Free memory
del candidates, texts, candidate_vectors, tokenized_corpus, model
import gc; gc.collect()
print("\n[DONE] All artifacts generated successfully!")

## 5. Scorer (Online Ranking - Under 5 Minutes)

This cell loads the pre-built artifacts and runs the 5-layer scoring engine.
This is the part that must run in under 5 minutes on a 16GB CPU-only machine.

In [ ]:
import re
import os
import pickle
import time
from datetime import datetime, timedelta
import numpy as np
import faiss


def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


# ── Honeypot Detection ────────────────────────────────────────────────────────
def is_honeypot(candidate):
    for skill in candidate.get("skills", []):
        if skill.get("proficiency") == "expert" and skill.get("duration_months", 1) == 0:
            return True

    expert_skills = [s for s in candidate.get("skills", []) if s.get("proficiency") == "expert"]
    if len(expert_skills) >= HONEYPOT_EXPERT_COUNT_THRESHOLD:
        durations = [s.get("duration_months", 0) for s in expert_skills]
        if sum(durations) / len(durations) < HONEYPOT_EXPERT_AVG_DURATION_MIN:
            return True

    history = candidate.get("career_history", [])
    total_career_months = sum(h.get("duration_months", 0) for h in history)
    stated_years = candidate.get("profile", {}).get("years_of_experience", 0)
    stated_months = stated_years * 12
    if stated_months > 0 and total_career_months > stated_months * HONEYPOT_EXPERIENCE_RATIO:
        return True

    for role in history:
        start_str = role.get("start_date")
        end_str = role.get("end_date")
        stated_duration = role.get("duration_months", 0)
        if start_str and end_str and stated_duration > 0:
            try:
                start = datetime.strptime(start_str, "%Y-%m-%d")
                end = datetime.strptime(end_str, "%Y-%m-%d")
                calculated = max(0, (end.year - start.year) * 12 + (end.month - start.month))
                if calculated > 0:
                    if abs(calculated - stated_duration) / calculated > HONEYPOT_DATE_MISMATCH_TOLERANCE:
                        return True
            except (ValueError, TypeError):
                pass

    dated_roles = []
    for role in history:
        start_str = role.get("start_date")
        end_str = role.get("end_date")
        if start_str:
            try:
                start = datetime.strptime(start_str, "%Y-%m-%d")
                end = datetime.strptime(end_str, "%Y-%m-%d") if end_str else datetime.now()
                dated_roles.append((start, end))
            except (ValueError, TypeError):
                pass
    dated_roles.sort(key=lambda x: x[0])
    for i in range(len(dated_roles) - 1):
        _, end_i = dated_roles[i]
        start_next, _ = dated_roles[i + 1]
        if (end_i - start_next).days > HONEYPOT_CAREER_OVERLAP_DAYS:
            return True

    return False


# ── Title Relevance ───────────────────────────────────────────────────────────
def title_relevance_score(candidate):
    title = candidate.get("profile", {}).get("current_title", "").lower().strip()
    if title in TITLE_TIER_PERFECT:
        return TITLE_SCORES["perfect"]
    if title in TITLE_TIER_STRONG_ADJACENT:
        return TITLE_SCORES["strong_adjacent"]
    if title in TITLE_TIER_WEAK_ADJACENT:
        return TITLE_SCORES["weak_adjacent"]
    return TITLE_SCORES["non_tech_trap"]


# ── Career Trajectory ─────────────────────────────────────────────────────────
def career_trajectory_score(candidate):
    score = 1.0
    history = candidate.get("career_history", [])
    if not history:
        return 0.5
    companies = [h.get("company", "").lower() for h in history]
    if all(any(firm in c for firm in CONSULTING_FIRMS) for c in companies):
        score *= 0.2
    product_months = 0
    for role in history:
        company = role.get("company", "").lower()
        if not any(firm in company for firm in CONSULTING_FIRMS):
            product_months += role.get("duration_months", 0)
    if product_months > 48:
        score *= 1.3
    elif product_months > 24:
        score *= 1.1
    elif product_months < 12:
        score *= 0.7
    return score


# ── Experience Range ──────────────────────────────────────────────────────────
def experience_range_score(candidate):
    yoe = candidate.get("profile", {}).get("years_of_experience", 0)
    for min_y, max_y, multiplier in EXPERIENCE_SCORES:
        if min_y <= yoe < max_y:
            return multiplier
    if yoe == 9:
        return 1.0
    return 0.3


# ── Location Score ────────────────────────────────────────────────────────────
def location_score(candidate):
    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})
    country = profile.get("country", "").strip().lower()
    location = profile.get("location", "").strip().lower()
    willing = signals.get("willing_to_relocate", False)
    is_india = country == "india"
    if is_india:
        if any(city in location for city in PREFERRED_CITIES_TIER1):
            return LOCATION_SCORES["india_tier1"]
        if any(city in location for city in PREFERRED_CITIES_TIER2):
            return LOCATION_SCORES["india_tier2"]
        return LOCATION_SCORES["india_other_relocate"] if willing else LOCATION_SCORES["india_other_no_relocate"]
    else:
        return LOCATION_SCORES["outside_india_relocate"] if willing else LOCATION_SCORES["outside_india_no_relocate"]


# ── Behavioral Multiplier ─────────────────────────────────────────────────────
def get_behavioral_multiplier(candidate):
    signals = candidate.get("redrob_signals", {})
    mult = 1.0
    response_rate = signals.get("recruiter_response_rate", 0.5)
    if response_rate < 0.1:
        mult *= 0.4
    elif response_rate < 0.2:
        mult *= 0.6
    last_active = signals.get("last_active_date")
    if last_active:
        try:
            days_inactive = (datetime.now() - datetime.strptime(last_active, "%Y-%m-%d")).days
            if days_inactive > 180:
                mult *= 0.5
            elif days_inactive > 90:
                mult *= 0.8
        except (ValueError, TypeError):
            pass
    return mult


# ── Final Score ───────────────────────────────────────────────────────────────
def compute_final_score(semantic_sim, bm25_sim, honeypot, title_sc, trajectory_sc, experience_sc, location_sc, behavioral_mult=1.0):
    if honeypot:
        return 0.001
    hybrid = SEMANTIC_WEIGHT * semantic_sim + BM25_WEIGHT * bm25_sim
    hybrid *= title_sc * trajectory_sc * experience_sc * location_sc * behavioral_mult
    return hybrid


# =============================================================================
# RUN SCORER
# =============================================================================

print("Loading artifacts ...")
start_time = time.time()

with open(CANDIDATES_META_PATH, "rb") as f:
    candidates = pickle.load(f)
index = faiss.read_index(DENSE_INDEX_PATH)
with open(BM25_INDEX_PATH, "rb") as f:
    bm25 = pickle.load(f)
jd_vector = np.load(JD_VECTOR_PATH)

n = len(candidates)
print(f"  Loaded {n} candidates, FAISS index, BM25 index")

# Semantic scores
print("Computing semantic scores ...")
D, I = index.search(jd_vector, n)
semantic_scores = np.zeros(n)
for score, idx in zip(D[0], I[0]):
    semantic_scores[idx] = score

# BM25 scores
print("Computing BM25 keyword scores ...")
tokenized_query = tokenize(BM25_JD_QUERY)
bm25_scores = bm25.get_scores(tokenized_query)
bm25_max = bm25_scores.max()
if bm25_max > 0:
    bm25_scores = bm25_scores / bm25_max

# Multi-layer scoring
print("Applying multi-layer scoring ...")
scored = []
honeypot_count = 0

for i, candidate in enumerate(candidates):
    hp = is_honeypot(candidate)
    if hp:
        honeypot_count += 1
    t_score = title_relevance_score(candidate)
    c_score = career_trajectory_score(candidate)
    e_score = experience_range_score(candidate)
    l_score = location_score(candidate)
    b_mult = get_behavioral_multiplier(candidate)
    f_score = compute_final_score(
        semantic_scores[i], bm25_scores[i], hp,
        t_score, c_score, e_score, l_score, b_mult
    )
    scored.append({
        "candidate_id": candidate["candidate_id"],
        "final_score": f_score,
        "semantic_score": float(semantic_scores[i]),
        "bm25_score": float(bm25_scores[i]),
        "is_honeypot": hp,
        "title_score": t_score,
        "trajectory_score": c_score,
        "experience_score": e_score,
        "location_score": l_score,
        "behavioral_mult": b_mult,
        "profile": candidate.get("profile", {}),
        "skills": candidate.get("skills", []),
        "redrob_signals": candidate.get("redrob_signals", {}),
    })

# Sort and normalize
scored.sort(key=lambda x: (-x["final_score"], x["candidate_id"]))

THEORETICAL_MAX_SCORE = 1.69
for c in scored:
    c["final_score"] = min(c["final_score"] / THEORETICAL_MAX_SCORE, 1.0)

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"  SCORING COMPLETED IN {elapsed:.1f} seconds")
print(f"  Total candidates scored: {n}")
print(f"  Honeypots detected: {honeypot_count}")
print(f"{'='*60}")

# Top 20 report
print(f"\n  Top 20 candidates:")
print(f"  {'Rank':<5} {'ID':<15} {'Score':>8} {'Sem':>6} {'KW':>6} {'Title':>6} {'Traj':>6} {'Exp':>5} {'Loc':>5} {'Beh':>5} HP")
for rank, c in enumerate(scored[:20], 1):
    print(
        f"  {rank:<5} {c['candidate_id']:<15} {c['final_score']:>8.4f} "
        f"{c['semantic_score']:>6.3f} {c['bm25_score']:>6.3f} "
        f"{c['title_score']:>6.2f} {c['trajectory_score']:>6.2f} "
        f"{c['experience_score']:>5.2f} {c['location_score']:>5.2f} "
        f"{c['behavioral_mult']:>5.2f} {'!HP' if c['is_honeypot'] else 'OK'}"
    )

# Honeypot safety check
top_100 = scored[:100]
hp_in_top = sum(1 for c in top_100 if c["is_honeypot"])
hp_rate = hp_in_top / len(top_100) * 100 if top_100 else 0
print(f"\n  Honeypot rate in top 100: {hp_rate:.1f}% {'-- DANGER!' if hp_rate > 10 else '(safe)'}")

## 6. Download Artifacts

Download the 4 artifact files to your local machine. Place them in your project root alongside `scorer.py`.

In [ ]:
from google.colab import files

for artifact in [DENSE_INDEX_PATH, BM25_INDEX_PATH, CANDIDATES_META_PATH, JD_VECTOR_PATH]:
    if os.path.exists(artifact):
        print(f"Downloading {artifact} ...")
        files.download(artifact)

print("\nAll artifacts downloaded! Place them in your project root.")